In [ ]:
from pathlib import Path

from plotting.viz import (
    build_downstream_peak_plateau_map,
    build_downstream_peak_plateau_map_statistical,
    build_embedding_cost_tables_by_dataset,
    build_hyperparameter_sensitivity_comparison_tables,
    build_script_lines,
    crawl_embedding_cost_results,
    crawl_downstream_accuracy_results,
    crawl_functional_results,
    crawl_hyperparameter_sensitivity_results,
    crawl_results,
    crawl_stability_performance_bootstrap_results,
    crawl_synth_results,
    filter_stability_performance_bootstrap_results,
    plot_downstream_mean_accuracy_lines,
    plot_downstream_mean_accuracy_lines_paper,
    plot_functional_mean_lines,
    plot_functional_mean_lines_paper,
    plot_hyperparameter_sensitivity_grids,
    plot_representational_mean_lines_paper,
    plot_representational_stability_with_performance_markers,
    plot_stability_performance_bootstrap_hit_rate_grid,
    plot_synth_representational_mean_lines_paper,
    report_leaf_value_status,
    SYNTH_LINE_COLOR_PRESETS,
    LINEPLOT_PAPER_STYLE_DEFAULT
)

from IPython.display import Markdown, display

from paths_globals import *

In [ ]:
REPSIM_MEASURE_RENAME_DICT = {
    "SecondOrderCosineSimilarity": "Second-Order Cosine Similarity",
    "JaccardSimilarity": "k-NN Jaccard Similarity",
    "AlignedCosineSimilarity": "Aligned Cosine Similarity",
    "DistanceCorrelation": "Distance Correlation",
}

FUNCSIM_MEASURE_RENAME_DICT = {
    "Disagreement": "Disagreement",
    "MinMaxNormalizedDisagreement": "Min-Max-Norm. Disagreement",
    "StableCore": "Stable Core",
    "JSD": "Jensen-Shannon Divergence",
}


In [ ]:
# Set up style for line plots
PAPER_LINE_STYLE = {
    **LINEPLOT_PAPER_STYLE_DEFAULT,
    "figure_width": 6.5,
    "subplot_height": 2.4,
    "tick_label_size": 7,
    "axis_label_size": 8,
    "title_size": 8,
    "title_pad": 1.0,
    "title_y": 1.02,
    "title_enumerate": True,
    "y_tick_step": 0.1,
    "tick_direction": "out",
    "tick_length": 4.8,
    "tick_width": 0.9,
    "tick_label_pad": 1.2,
    "x_label_pad": 1.5,
    "y_label_pad": 5.0,
    "legend_bbox_y": 0.05,
    "x_tick_rotation": 45,
    "x_tick_ha": "right",
    "x_tick_rotation_mode": "anchor",
    "figure_facecolor": "white",
    "axes_facecolor": "white",
}


## 1 Downstream Performance (run this first)

Evaluate performance across dimensions first, and store best/near-best dimensions for use in later stability plots.


In [ ]:
# Crawl downstream means (used for quick inspection/debugging).
perf_results = crawl_downstream_accuracy_results()


In [ ]:
PERFORMANCE_MARKER_MODE = "threshold"  # one of: "none", "threshold", "statistical"
INCLUDE_CONFIDENCE_BANDS = True
CONFIDENCE_BAND_MODE = "std"
for clf_name in [LOGISTIC_REGRESSION, MULTILAYER_PERCEPTRON]:
    plot_downstream_mean_accuracy_lines_paper(
        perf_results=perf_results,
        classifier_name=clf_name,
        datasets=[CORA, PUBMED, BLOGCATALOG, FACEBOOK, WIKIPEDIA, COAUTHOR, DDI],
        metric_name=ACCURACY_SCORE,
        style=PAPER_LINE_STYLE,
        output_dir=Path(PLOTS_DIR) / f"{PERFORMANCE_MARKER_MODE}_performance_markers",
        show=False,
        show_performance_markers=True,
        performance_marker_mode=PERFORMANCE_MARKER_MODE,
        y_axis_mode="manual",
        y_axis_limits=(0.2,1),
        include_confidence_bands=INCLUDE_CONFIDENCE_BANDS,
        confidence_band_mode=CONFIDENCE_BAND_MODE,
    )


In [ ]:
# Configure downstream marker selection. Maps are computed lazily by mode and classifier.
PERFORMANCE_MARKER_CLASSIFIER = LOGISTIC_REGRESSION
PERFORMANCE_MARKER_ALPHA = 0.05
PERFORMANCE_MARKER_MIN_PLATEAU_SIZE = 2
PERFORMANCE_MARKER_RELATIVE_TOLERANCE = 0.01

perf_peak_plateau_maps = {("none", None): None}


def get_perf_peak_plateau_map(marker_mode, classifier_name=PERFORMANCE_MARKER_CLASSIFIER):
    if marker_mode not in {"none", "threshold", "statistical"}:
        raise ValueError("marker_mode must be one of: 'none', 'threshold', 'statistical'")
    if marker_mode == "none":
        return None

    cache_key = (marker_mode, classifier_name)
    if cache_key not in perf_peak_plateau_maps:
        if marker_mode == "threshold":
            perf_peak_plateau_maps[cache_key] = build_downstream_peak_plateau_map(
                perf_results=perf_results,
                relative_tolerance=PERFORMANCE_MARKER_RELATIVE_TOLERANCE,
                classifier_name=classifier_name,
                min_plateau_size=PERFORMANCE_MARKER_MIN_PLATEAU_SIZE,
            )
        elif marker_mode == "statistical":
            perf_peak_plateau_maps[cache_key] = build_downstream_peak_plateau_map_statistical(
                classifier_name=classifier_name,
                metric=ACCURACY_SCORE,
                alpha=PERFORMANCE_MARKER_ALPHA,
            )
    return perf_peak_plateau_maps[cache_key]

# Backward-compatible alias; remains None until a plotting cell selects a marker mode.
perf_peak_plateau_map = None


In [ ]:
# Overview plot: manually switch marker mode here.
PERFORMANCE_MARKER_MODE = "threshold"  # one of: "none", "threshold", "statistical"
INCLUDE_CONFIDENCE_BANDS = True
CONFIDENCE_BAND_MODE = "std"
plot_downstream_mean_accuracy_lines(
    perf_results=perf_results,
    datasets=[CORA, PUBMED, BLOGCATALOG, FACEBOOK, WIKIPEDIA, COAUTHOR, DDI],
    perf_peak_plateau_map=get_perf_peak_plateau_map(PERFORMANCE_MARKER_MODE),
    performance_marker_mode=PERFORMANCE_MARKER_MODE,
    performance_marker_classifier_name=PERFORMANCE_MARKER_CLASSIFIER,
    legend_position="bottom",
    include_confidence_bands=INCLUDE_CONFIDENCE_BANDS,
    confidence_band_mode=CONFIDENCE_BAND_MODE,
)


## 2 Representational Stability

Representational similarity over embedding dimension.


In [ ]:
# Crawl representational stability means (used for quick inspection/debugging).
results = crawl_results()


In [ ]:
algorithms = sorted(results.keys())
similarity_measures = sorted({sim for algo in results.values() for sim in algo.keys()})
datasets = sorted({dataset for algo in results.values() for sim in algo.values() for dataset in sim.keys()})


In [ ]:
# Export one paper-ready PDF per representational stability measure row.
SHOW_PERFORMANCE_MARKERS = True
PERFORMANCE_MARKER_MODE = "threshold"  # one of: "none", "threshold", "statistical"
INCLUDE_CONFIDENCE_BANDS = True
CONFIDENCE_BAND_MODE = "std"
Y_AXIS_MODE = "zoom"  # one of: "fixed", "zoom", "manual"
Y_AXIS_LIMITS = None  # tuple like (0.85, 1.01) or list of per-panel tuples in manual mode
Y_AXIS_PADDING = 0.02
for measure_name in sorted({m for algo_map in results.values() for m in algo_map.keys()}):
    plot_representational_mean_lines_paper(
        results=results,
        measure_name=measure_name,
        datasets=[CORA, PUBMED, BLOGCATALOG, FACEBOOK, WIKIPEDIA, COAUTHOR, DDI],
        style=PAPER_LINE_STYLE,
        output_dir=Path(PLOTS_DIR) / f"{PERFORMANCE_MARKER_MODE}_performance_markers",
        show=False,
        show_performance_markers=SHOW_PERFORMANCE_MARKERS,
        perf_peak_plateau_map=get_perf_peak_plateau_map(PERFORMANCE_MARKER_MODE),
        performance_marker_mode=PERFORMANCE_MARKER_MODE,
        y_axis_mode=Y_AXIS_MODE,
        y_axis_limits=Y_AXIS_LIMITS,
        y_axis_padding=Y_AXIS_PADDING,
        include_confidence_bands=INCLUDE_CONFIDENCE_BANDS,
        confidence_band_mode=CONFIDENCE_BAND_MODE,
    )


In [ ]:
plot_representational_mean_lines_paper(
        results=results,
        measure_name="AlignedCosineSimilarity",
        datasets=[CORA, PUBMED, BLOGCATALOG, FACEBOOK, WIKIPEDIA, COAUTHOR, DDI],
        style=PAPER_LINE_STYLE,
        output_dir=Path(PLOTS_DIR) / f"{PERFORMANCE_MARKER_MODE}_performance_markers",
        show=False,
        show_performance_markers=SHOW_PERFORMANCE_MARKERS,
        perf_peak_plateau_map=get_perf_peak_plateau_map("threshold"),
        performance_marker_mode="threshold",
        y_axis_mode="manual",
        y_axis_limits=[(.5,1), (.5,1), (.2,1), (.8,1), (.2,1)],
        y_axis_padding=Y_AXIS_PADDING,
        include_confidence_bands=INCLUDE_CONFIDENCE_BANDS,
        confidence_band_mode=CONFIDENCE_BAND_MODE,
    )


In [ ]:
plot_representational_mean_lines_paper(
        results=results,
        measure_name="JaccardSimilarity",
        datasets=[CORA, PUBMED, BLOGCATALOG, FACEBOOK, WIKIPEDIA, COAUTHOR, DDI],
        style=PAPER_LINE_STYLE,
        output_dir=Path(PLOTS_DIR) / f"{PERFORMANCE_MARKER_MODE}_performance_markers",
        show=False,
        show_performance_markers=SHOW_PERFORMANCE_MARKERS,
        perf_peak_plateau_map=get_perf_peak_plateau_map("threshold"),
        performance_marker_mode="threshold",
        y_axis_mode="manual",
        y_axis_limits=[(0,.8), (0,.6), (0,.5), (0,1), (0,.4)],
        y_axis_padding=Y_AXIS_PADDING,
        include_confidence_bands=INCLUDE_CONFIDENCE_BANDS,
        confidence_band_mode=CONFIDENCE_BAND_MODE,
    )


In [ ]:
plot_representational_mean_lines_paper(
        results=results,
        measure_name="DistanceCorrelation",
        datasets=[CORA, PUBMED, BLOGCATALOG, FACEBOOK, WIKIPEDIA, COAUTHOR, DDI],
        style=PAPER_LINE_STYLE,
        output_dir=Path(PLOTS_DIR) / f"{PERFORMANCE_MARKER_MODE}_performance_markers",
        show=False,
        show_performance_markers=SHOW_PERFORMANCE_MARKERS,
        perf_peak_plateau_map=get_perf_peak_plateau_map("threshold"),
        performance_marker_mode="threshold",
        y_axis_mode="manual",
        y_axis_limits=[(.3,1), (0,1), (.3,1), (.8,1), (0,1)],
        y_axis_padding=Y_AXIS_PADDING,
        include_confidence_bands=INCLUDE_CONFIDENCE_BANDS,
        confidence_band_mode=CONFIDENCE_BAND_MODE,
    )

In [ ]:
# Overlay best/near-best downstream dimensions on stability curves.
PERFORMANCE_MARKER_MODE = "threshold"  # one of: "none", "threshold", "statistical"
INCLUDE_CONFIDENCE_BANDS = True
plot_representational_stability_with_performance_markers(
    results=results,
    perf_peak_plateau_map=get_perf_peak_plateau_map(PERFORMANCE_MARKER_MODE),
    performance_marker_mode=PERFORMANCE_MARKER_MODE,
    algorithm_axis="columns",
    datasets=[CORA, PUBMED, BLOGCATALOG, FACEBOOK, WIKIPEDIA],
    include_confidence_bands=INCLUDE_CONFIDENCE_BANDS,
    confidence_band_mode=CONFIDENCE_BAND_MODE,
)


## 3 Functional Similarity

Functional similarity over embedding dimension, with separate grids per downstream classifier and columns as measures.


In [ ]:
# Crawl functional means (used for quick inspection/debugging).
functional_results = crawl_functional_results()

In [ ]:
# Hack: exclude ASNE functional-similarity results on DDI and CoAuthor from plots.
EXCLUDED_FUNCTIONAL_RESULTS = {ASNE: {DDI, COAUTHOR}}
removed_functional_entries = 0
for clf_map in functional_results.values():
    for algo, excluded_datasets in EXCLUDED_FUNCTIONAL_RESULTS.items():
        if algo not in clf_map:
            continue
        for measure_map in clf_map[algo].values():
            for dataset in excluded_datasets:
                if measure_map.pop(dataset, None) is not None:
                    removed_functional_entries += 1

print(f"Removed {removed_functional_entries} ASNE functional result entries for DDI/CoAuthor.")


In [ ]:
# Overview plot: manually switch marker mode here.
PERFORMANCE_MARKER_MODE = "threshold"  # one of: "none", "threshold", "statistical"
INCLUDE_CONFIDENCE_BANDS = True
CONFIDENCE_BAND_MODE = "std"
Y_AXIS_SCALE = "linear"  # one of: "linear", "log", "symlog"
Y_AXIS_SYMLOG_LINTHRESH = 0.0001  # scalar, list in algorithm order, or dict keyed by algorithm
Y_AXIS_SYMLOG_LINSCALE = 0.45  # makes 0-to-linthresh about half a log-decade
plot_functional_mean_lines(
    functional_results=functional_results,
    datasets=[CORA, PUBMED, BLOGCATALOG, FACEBOOK, WIKIPEDIA, COAUTHOR],
    performance_marker_mode=PERFORMANCE_MARKER_MODE,
    legend_position="bottom",
    y_axis_scale=Y_AXIS_SCALE,
    y_axis_symlog_linthresh=Y_AXIS_SYMLOG_LINTHRESH,
    y_axis_symlog_linscale=Y_AXIS_SYMLOG_LINSCALE,
    include_confidence_bands=INCLUDE_CONFIDENCE_BANDS,
    confidence_band_mode=CONFIDENCE_BAND_MODE,
)


In [ ]:
# Export one paper-ready PDF per functional row (classifier + measure).
SHOW_PERFORMANCE_MARKERS = True
PERFORMANCE_MARKER_MODE = "threshold"  # one of: "none", "threshold", "statistical"
INCLUDE_CONFIDENCE_BANDS = True
CONFIDENCE_BAND_MODE = "std"
Y_AXIS_MODE = "zoom"  # one of: "fixed", "zoom", "manual"
Y_AXIS_SCALE = "linear"  # one of: "linear", "log", "symlog"
Y_AXIS_SYMLOG_LINTHRESH = 0.0001  # scalar, list in algorithm order, or dict keyed by algorithm
Y_AXIS_SYMLOG_LINSCALE = 0.45  # makes 0-to-linthresh about half a log-decade

Y_AXIS_LIMITS = None  # tuple like (0.0, 0.2) or list of per-panel tuples in manual mode
Y_AXIS_PADDING = 0.02
for clf_name in sorted(functional_results.keys()):
    measure_names = sorted({m for algo_map in functional_results[clf_name].values() for m in algo_map.keys()})
    for measure_name in measure_names:
        plot_functional_mean_lines_paper(
            functional_results=functional_results,
            classifier_name=clf_name,
            measure_name=measure_name,
            datasets=[CORA, PUBMED, BLOGCATALOG, FACEBOOK, WIKIPEDIA, COAUTHOR, DDI],
            style=PAPER_LINE_STYLE,
            output_dir=Path(PLOTS_DIR) / f"{PERFORMANCE_MARKER_MODE}_performance_markers",
            show=False,
            show_performance_markers=SHOW_PERFORMANCE_MARKERS,
            performance_marker_mode=PERFORMANCE_MARKER_MODE,
            y_axis_mode=Y_AXIS_MODE,
            y_axis_limits=Y_AXIS_LIMITS,
            y_axis_padding=Y_AXIS_PADDING,
            y_axis_scale=Y_AXIS_SCALE,
            y_axis_symlog_linthresh=Y_AXIS_SYMLOG_LINTHRESH,
            y_axis_symlog_linscale=Y_AXIS_SYMLOG_LINSCALE,
            include_confidence_bands=INCLUDE_CONFIDENCE_BANDS,
            confidence_band_mode=CONFIDENCE_BAND_MODE,
        )


In [ ]:
# Export one paper-ready PDF per functional row (classifier + measure).
SHOW_PERFORMANCE_MARKERS = True
PERFORMANCE_MARKER_MODE = "threshold"  # one of: "none", "threshold", "statistical"
INCLUDE_CONFIDENCE_BANDS = True
Y_AXIS_MODE = "zoom"  # one of: "fixed", "zoom", "manual"
Y_AXIS_SCALE = "symlog"  # one of: "linear", "log", "symlog"
Y_AXIS_SYMLOG_LINTHRESH = 0.0001  # scalar, list in algorithm order, or dict keyed by algorithm
Y_AXIS_SYMLOG_LINSCALE = 0.45  # makes 0-to-linthresh about half a log-decade

Y_AXIS_LIMITS = None  # tuple like (0.0, 0.2) or list of per-panel tuples in manual mode
Y_AXIS_PADDING = 0.02

plot_functional_mean_lines_paper(
    functional_results=functional_results,
    classifier_name="LogisticRegression",
    measure_name="JSD",
    datasets=[CORA, PUBMED, BLOGCATALOG, FACEBOOK, WIKIPEDIA, COAUTHOR, DDI],
    style=PAPER_LINE_STYLE,
    output_dir=Path(PLOTS_DIR) / f"{PERFORMANCE_MARKER_MODE}_performance_markers",
    show=False,
    show_performance_markers=SHOW_PERFORMANCE_MARKERS,
    performance_marker_mode=PERFORMANCE_MARKER_MODE,
    y_axis_mode="manual",
    y_axis_limits=(0, 0.1),
    y_axis_padding=Y_AXIS_PADDING,
    y_axis_scale=Y_AXIS_SCALE,
    y_axis_symlog_linthresh=[.0001, .001, .001, .0001, .0001],
    y_axis_symlog_linscale=Y_AXIS_SYMLOG_LINSCALE,
    include_confidence_bands=INCLUDE_CONFIDENCE_BANDS,
    confidence_band_mode=CONFIDENCE_BAND_MODE,
)

plot_functional_mean_lines_paper(
    functional_results=functional_results,
    classifier_name="MLP",
    measure_name="JSD",
    datasets=[CORA, PUBMED, BLOGCATALOG, FACEBOOK, WIKIPEDIA, COAUTHOR, DDI],
    style=PAPER_LINE_STYLE,
    output_dir=Path(PLOTS_DIR) / f"{PERFORMANCE_MARKER_MODE}_performance_markers",
    show=False,
    show_performance_markers=SHOW_PERFORMANCE_MARKERS,
    performance_marker_mode=PERFORMANCE_MARKER_MODE,
    y_axis_mode="manual",
    y_axis_limits=(0, 0.1),
    y_axis_padding=Y_AXIS_PADDING,
    y_axis_scale=Y_AXIS_SCALE,
    y_axis_symlog_linthresh=[.0001, .001, .001, .001, .001],
    y_axis_symlog_linscale=Y_AXIS_SYMLOG_LINSCALE,
    include_confidence_bands=INCLUDE_CONFIDENCE_BANDS,
    confidence_band_mode=CONFIDENCE_BAND_MODE,
)


## 4 Synthetic Representational Similarity

Representational similarity over embedding dimension for synthetic datasets, with tonal lines for varying size or density.


In [ ]:
# Crawl synthetic representational means (used for quick inspection/debugging).
synth_results = crawl_synth_results()

synth_algorithms = sorted(synth_results.keys())
synth_measures = sorted({m for algo_map in synth_results.values() for m in algo_map.keys()})
synth_datasets = sorted({ds for algo_map in synth_results.values() for m_map in algo_map.values() for ds in m_map.keys()})

synth_algorithms, synth_measures, synth_datasets


In [ ]:
# Export paper-ready synthetic representational plots for all dataset/measure/mode combinations.
SYNTH_PLOT_DATASETS = sorted({ds for algo_map in synth_results.values() for m_map in algo_map.values() for ds in m_map.keys()})
SYNTH_PLOT_MEASURES = sorted({m for algo_map in synth_results.values() for m in algo_map.keys()})
SYNTH_PLOT_MODES = ["size", "density"]

# Fixed counterpart values used by the two variation modes.
FIXED_DENSITY = SYNTH_DATA_EXPERIMENTS_DEFAULT_DENSITY
FIXED_NUM_NODES = SYNTH_DATA_EXPERIMENTS_DEFAULT_NUM_NODES
print(f"Datasets: {SYNTH_PLOT_DATASETS}")
print(f"Measures: {SYNTH_PLOT_MEASURES}")
print(f"Modes: {SYNTH_PLOT_MODES}")
print(f"Color presets: {sorted(SYNTH_LINE_COLOR_PRESETS.keys())}")
SYNTH_COLOR_SCHEME = "viridis_mid_dark"
INCLUDE_CONFIDENCE_BANDS = True
CONFIDENCE_BAND_MODE = "std"

for dataset_name in SYNTH_PLOT_DATASETS:
    for measure_name in SYNTH_PLOT_MEASURES:
        for mode in SYNTH_PLOT_MODES:
            vary_size = mode == "size"
            vary_density = mode == "density"

            print(f"Plotting dataset={dataset_name}, measure={measure_name}, mode={mode}")
            plot_synth_representational_mean_lines_paper(
                synth_results=synth_results,
                dataset=dataset_name,
                measure_name=measure_name,
                vary_size=vary_size,
                vary_density=vary_density,
                fixed_density=FIXED_DENSITY,
                fixed_num_nodes=FIXED_NUM_NODES,
                color_scheme=SYNTH_COLOR_SCHEME,
                style=PAPER_LINE_STYLE,
                output_dir=PLOTS_DIR,
                show=False,
                y_axis_mode="zoom",
                y_axis_limits=None,
                y_axis_padding=0.02,
                include_confidence_bands=INCLUDE_CONFIDENCE_BANDS,
                confidence_band_mode=CONFIDENCE_BAND_MODE,
            )


## 5 Stability-Performance Bootstrap


In [ ]:
bootstrap_results = crawl_stability_performance_bootstrap_results()
bootstrap_results_filtered = filter_stability_performance_bootstrap_results(
    bootstrap_results,
    excluded_algorithm_datasets={ASNE: {DDI, COAUTHOR}},
)
print(
    f"Loaded {len(bootstrap_results)} bootstrap summary rows; "
    f"kept {len(bootstrap_results_filtered)} after excluding ASNE on DDI/CoAuthor."
)


In [ ]:
BOOTSTRAP_HIT_RATE_MEASURES = [
    "AlignedCosineSimilarity",
    "JaccardSimilarity",
    "Disagreement",
    "JSD"
]
# BOOTSTRAP_HIT_RATE_MEASURES = [
#     "DistanceCorrelation",
#     "SecondOrderCosineSimilarity",
#     "MinMaxNormalizedDisagreement",
# ]
BOOTSTRAP_PERFORMANCE_CRITERION = "strict_best"  # one of: "threshold", "statistical", "strict_best"

plot_stability_performance_bootstrap_hit_rate_grid(
    BOOTSTRAP_HIT_RATE_MEASURES,
    results=bootstrap_results_filtered,
    classifier=LOGISTIC_REGRESSION,
    performance_criterion=BOOTSTRAP_PERFORMANCE_CRITERION,
    output_dir=Path(PLOTS_DIR) / "bootstrap_hit_rates",
    save=True,
    show=False,
)


## 6 Embedding Costs


In [ ]:
res_dir = Path(OUTPUT_DIR) / "embedding_costs" / "helix"
embedding_cost_results = crawl_embedding_cost_results(results_dirs=res_dir, run_ids="all")
embedding_cost_tables = build_embedding_cost_tables_by_dataset(
    embedding_cost_results,
    classifier=LOGISTIC_REGRESSION,
    dimensions=None,
    time_unit="s",
    memory_unit="GB",
    memory_field="peak_rss_delta_bytes_mean",
)

print(
    f"Loaded {len(embedding_cost_results['embedding'])} embedding-cost summary rows "
    f"and {len(embedding_cost_results['downstream'])} downstream-cost summary rows."
)


In [ ]:
for dataset_name, table in embedding_cost_tables.items():
    display(Markdown(f"### {DATASET_RENAME_DICT.get(dataset_name, dataset_name)}"))
    display(table.style.format("{:.2f}", na_rep="--"))


In [ ]:
embedding_cost_table_dir = Path(TABLES_DIR) / "embedding_costs"
embedding_cost_table_dir.mkdir(parents=True, exist_ok=True)

for dataset_name, table in embedding_cost_tables.items():
    out_path = embedding_cost_table_dir / f"embedding_costs_{dataset_name}.tex"
    caption = f"Embedding and downstream evaluation costs for {DATASET_RENAME_DICT.get(dataset_name, dataset_name)}."
    label = f"tab:embedding-costs-{dataset_name}".replace("_", "-")
    latex = table.to_latex(
        na_rep="--",
        float_format=lambda value: f"{value:.2f}",
        escape=False,
        multicolumn=True,
        multicolumn_format="c",
        multirow=True,
        caption=caption,
        label=label,
        position="htbp",
    )
    out_path.write_text(latex, encoding="utf-8")
    print(f"Saved {out_path}")


## 7 Hyperparameter Sensitivity


In [ ]:
HYPERPARAMETER_SENSITIVITY_DATASET = WIKIPEDIA
HYPERPARAMETER_SENSITIVITY_CLASSIFIER = LOGISTIC_REGRESSION
HYPERPARAMETER_SENSITIVITY_MEASURES = [
    "JaccardSimilarity",
    "AlignedCosineSimilarity",
    "Disagreement",
    "JSD",
]

hyperparameter_sensitivity_results = crawl_hyperparameter_sensitivity_results(
    datasets=[HYPERPARAMETER_SENSITIVITY_DATASET],
)
hyperparameter_sensitivity_tables = build_hyperparameter_sensitivity_comparison_tables(
    hyperparameter_sensitivity_results,
    datasets=[HYPERPARAMETER_SENSITIVITY_DATASET],
    classifier=HYPERPARAMETER_SENSITIVITY_CLASSIFIER,
    performance_metric=ACCURACY_SCORE,
    stability_measures=HYPERPARAMETER_SENSITIVITY_MEASURES,
    measure_labels={
        ACCURACY_SCORE: "Accuracy",
        **REPSIM_MEASURE_RENAME_DICT,
        **FUNCSIM_MEASURE_RENAME_DICT,
    },
    include_deltas=False,
    only_tuned_when_params_changed=False,
    only_changed_or_improved=True,
    validation_gain_threshold=0.0,
    column_layout="metric_first",
    stage1_columns="validation_scores",
    round_digits=3,
)

print({key: len(value) for key, value in hyperparameter_sensitivity_results.items()})


In [ ]:
for algorithm_name, table in hyperparameter_sensitivity_tables.items():
    display(Markdown(f"### {EMBEDDING_ALGORITHM_RENAME_DICT.get(algorithm_name, algorithm_name)}"))
    display(table.style.format(precision=3, na_rep="--"))


In [ ]:
hyperparameter_sensitivity_table_dir = Path(TABLES_DIR) / "hyperparameter_sensitivity"
hyperparameter_sensitivity_table_dir.mkdir(parents=True, exist_ok=True)

dataset_label = DATASET_RENAME_DICT.get(HYPERPARAMETER_SENSITIVITY_DATASET, HYPERPARAMETER_SENSITIVITY_DATASET)
classifier_label = HYPERPARAMETER_SENSITIVITY_CLASSIFIER

for algorithm_name, table in hyperparameter_sensitivity_tables.items():
    out_path = hyperparameter_sensitivity_table_dir / f"hyperparameter_sensitivity_{HYPERPARAMETER_SENSITIVITY_DATASET}_{algorithm_name}_{HYPERPARAMETER_SENSITIVITY_CLASSIFIER}.tex"
    algorithm_label = EMBEDDING_ALGORITHM_RENAME_DICT.get(algorithm_name, algorithm_name)
    caption = f"Hyperparameter sensitivity results for {algorithm_label} on {dataset_label} using {classifier_label}."
    label = f"tab:hyperparameter-sensitivity-{HYPERPARAMETER_SENSITIVITY_DATASET}-{algorithm_name}-{HYPERPARAMETER_SENSITIVITY_CLASSIFIER}".replace("_", "-")
    latex = table.to_latex(
        na_rep="--",
        float_format=lambda value: f"{value:.3f}",
        escape=False,
        multicolumn=True,
        multicolumn_format="c",
        multirow=True,
        caption=caption,
        label=label,
        position="htbp",
    )
    out_path.write_text(latex, encoding="utf-8")
    print(f"Saved {out_path}")


In [ ]:
HYPERPARAMETER_SENSITIVITY_PLOT_STYLE = {
    **PAPER_LINE_STYLE,
    "max_cols": 2,
    "figure_width": 6.5,
    "subplot_width": 3.0,
    "subplot_height": 1.8,
    "row_height_pad": 0.52,
    "tight_layout_bottom": 0.06,
    "tight_layout_top": 0.91,
    "hyperparameter_suptitle_y": 0.875,
    "hyperparameter_legend_position": "side",
    "hyperparameter_legend_bbox": (0.74, 0.24),
    "hyperparameter_show_y_label": False,
}

hyperparameter_sensitivity_plot_paths = plot_hyperparameter_sensitivity_grids(
    hyperparameter_sensitivity_results,
    dataset=HYPERPARAMETER_SENSITIVITY_DATASET,
    classifier=HYPERPARAMETER_SENSITIVITY_CLASSIFIER,
    performance_metric=ACCURACY_SCORE,
    stability_measures=HYPERPARAMETER_SENSITIVITY_MEASURES,
    measure_labels={
        ACCURACY_SCORE: "Accuracy",
        **REPSIM_MEASURE_RENAME_DICT,
        **FUNCSIM_MEASURE_RENAME_DICT,
    },
    dimensions=EXPERIMENTS_DIMENSIONS_LIST,
    style=HYPERPARAMETER_SENSITIVITY_PLOT_STYLE,
    include_variance_bands=True,
    variance_band_mode="std",
    tuned_only_when_params_changed=False,
    connect_tuned_segments_to_reference=True,
    jsd_symlog_linthresh=1e-4,
    output_dir=Path(PLOTS_DIR) / "hyperparameter_sensitivity",
    show=False,
    save=True,
)


## Diagnostics


In [ ]:
# Diagnostic helper moved to plots/viz.py.


In [ ]:
# Parsing/script-generation helper functions moved to plots/viz.py.


In [ ]:
leaf_issues = report_leaf_value_status()

script_lines = build_script_lines(issues=leaf_issues, n_jobs=32)
# out_path = Path(args.output_script)
# out_path.parent.mkdir(parents=True, exist_ok=True)
# out_path.write_text("\n".join(script_lines) + "\n", encoding="utf-8")
# print(f"Wrote {len(issues)} rerun lines to {out_path}")


In [ ]:
script_lines

In [ ]:
out_path = Path("shell_scripts/repsim_rerun.sh")
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text("\n".join(script_lines) + "\n", encoding="utf-8")
# print(f"Wrote {len(issues)} rerun lines to {out_path}")
